# Dataset Mention Extraction Pipeline Demo

This notebook demonstrates how to use the `ai4data` library to extract dataset mentions, identify their attributes (acronyms, producers, timeframes, geographies), and classify their typology and usage contexts.

## Pipeline Architecture
1. **Pre-filtering (Sequence Classification)**: A page/text is processed by `ai4data/datause-classifier` (a GLiNER2-based sequence classification adapter). If the classifier predicts `NO_DATA`, the page is skipped immediately.
2. **Named Entity & Relation Extraction**: If the page contains data references, the GLiNER2 base model (`fastino/gliner2-large-v1`) with the LoRA adapter (`ai4data/datause-extraction`) extracts dataset mentions and links them to acronyms, organizations, timeframes, and geographies.
3. **Zero-Shot Classification**: Extracted mentions are classified in Pass 2 to determine:
   - **Typology**: `survey`, `census`, `database`, `administrative`, `indicator`, etc.
   - **Usage Context**: `primary`, `supporting`, `background`.

### 1. Setup Environment and Imports

In [ ]:
import os
import sys
import pandas as pd

# Add src/ to path if running locally
src_path = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
if os.path.exists(src_path) and src_path not in sys.path:
    sys.path.insert(0, src_path)

from ai4data import extract_from_text, extract_from_document
print("ai4data loaded successfully!")

### 2. Text-Level Extraction (`extract_from_text`)

Let's run a simple extraction on a text paragraph containing multiple dataset mentions. We will enable the pre-filtering classifier and show the resolved attributes.

In [ ]:
text = (
    "We conduct our main analysis using microdata from the 2018 Nigeria General Household Survey (GHS), "
    "which was conducted by the National Bureau of Statistics (NBS) in collaboration with the World Bank. "
    "The GHS contains detailed survey responses across 36 states in Nigeria. "
    "To complement these findings and check macroeconomic trends, we integrate indicators from the "
    "World Development Indicators (WDI) compiled by the World Bank database covering the years 2010 to 2020. "
    "Finally, as a preliminary reference, we mention population counts from the 2006 Population and "
    "Housing Census of Nigeria, published by the National Population Commission."
)

result = extract_from_text(
    text,
    use_classifier=True,
    include_confidence=True,
)

# Format results as a DataFrame
rows = []
for ds in result.get("datasets", []):
    rows.append({
        "Mention Name": ds["mention_name"]["text"],
        "Confidence": f"{ds['mention_name']['confidence']:.3f}",
        "Acronym": ds["acronym"]["text"] or "None",
        "Producer": ds["producer"]["text"] or "None",
        "Geography": ds["geography"]["text"] or "None",
        "Typology": ds["typology_tag"]["text"],
        "Usage Context": ds["usage_context"]["text"]
    })

df_text = pd.DataFrame(rows)
df_text

### 3. Document-Level Extraction (`extract_from_document`)

Now let's run the extraction on pages 0 to 5 of the Ghana gold mining PDF document.

In [ ]:
pdf_path = "/Users/rafaelmacalaba/WBG/monitoring_of_datause/datause_extract/ghana_gold_mining.pdf"

results = extract_from_document(
    pdf_path,
    pages=[0, 1, 2, 3, 4, 5],
    use_classifier=True,
    include_confidence=True,
)

print(f"Processed {len(results)} chunks.")

### 4. Display Page-by-Page Extraction & Classifier Summary

In [ ]:
for res in results:
    print(f"\n=== Chunk {res['chunk']} (Page {res['page'] + 1}) ===")
    if res["classifier_skipped"]:
        print(f"  [SKIPPED] Pre-filter determined no dataset mentions present (Reason: {res['skip_reason']})")
    else:
        print(f"  [PROCESSED] Extracted {len(res['datasets'])} datasets:")
        for ds in res["datasets"]:
            print(f"    - {ds['mention_name']['text']}")
            print(f"      * Acronym:  {ds['acronym']['text'] or 'None'}")
            print(f"      * Producer: {ds['producer']['text'] or 'None'}")
            print(f"      * Typology: {ds['typology_tag']['text']}")
            print(f"      * Usage:    {ds['usage_context']['text']}")